#**스마트폰 센서 데이터 기반 모션 분류**
# 단계2 : 기본 모델링


## 0.미션

* 데이터 전처리
    * 가변수화, 데이터 분할, NaN 확인 및 조치, 스케일링 등 필요한 전처리 수행
* 다양한 알고리즘으로 분류 모델 생성
    * 최소 4개 이상의 알고리즘을 적용하여 모델링 수행
    * 성능 비교
    * 각 모델의 성능을 저장하는 별도 데이터 프레임을 만들고 비교
* 옵션 : 다음 사항은 선택사항입니다. 시간이 허용하는 범위 내에서 수행하세요.
    * 상위 N개 변수를 선정하여 모델링 및 성능 비교
        * 모델링에 항상 모든 변수가 필요한 것은 아닙니다.
        * 변수 중요도 상위 N개를 선정하여 모델링하고 타 모델과 성능을 비교하세요.
        * 상위 N개를 선택하는 방법은, 변수를 하나씩 늘려가며 모델링 및 성능 검증을 수행하여 적절한 지점을 찾는 것입니다.

## 1.환경설정

In [1]:
import random
import numpy as np
import os

# 1. 파이썬 자체 난수 고정
random.seed(42)

# 2. 넘파이 난수 고정
np.random.seed(42)

# 3. 텐서플로우 사용하는 경우 (딥러닝)
import tensorflow as tf
tf.random.set_seed(42)

# 4. (선택) 실행 환경 내에서 다른 난수도 통제하고 싶을 때
os.environ['PYTHONHASHSEED'] = '42'

In [2]:
# # 한글 깨짐 방지 위해 구글 코랩에 나눔폰트 설치.
# import sys
# if 'google.colab' in sys.modules:
#     !echo 'debconf debconf/frontend select Noninteractive' | debconf-set-selections
#     !sudo apt-get -qq -y install fonts-nanum
#     import matplotlib.font_manager as fm
#     font_files = fm.findSystemFonts(fontpaths=['/usr/share/fonts/truetype/nanum'])
#     for fpath in font_files:
#         fm.fontManager.addfont(fpath)

In [3]:
# # 나눔고딕으로 폰트 변경
# import matplotlib.pyplot as plt
# plt.rcParams['font.family'] = 'NanumGothic'

In [4]:
# # matplot의 폰트와 폰트 크기 확인
# print(plt.rcParams['font.family'], plt.rcParams['font.size'])

### (1) 라이브러리 불러오기

* 세부 요구사항
    - 기본적으로 필요한 라이브러리를 import 하도록 코드가 작성되어 있습니다.
    - 필요하다고 판단되는 라이브러리를 추가하세요.

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 필요하다고 판단되는 라이브러리를 추가하세요.
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import LabelEncoder
# from xgboost import XGBClassifier
# import lightgbm as lgb
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split, cross_validate
from sklearn.metrics import *
import joblib

* 함수 생성

In [6]:
# 변수의 특성 중요도 계산하기
def plot_feature_importance(importance, names, result_only = False, topn = 'all'):
    feature_importance = np.array(importance)
    feature_name = np.array(names)

    data={'feature_name':feature_name,'feature_importance':feature_importance}
    fi_temp = pd.DataFrame(data)

    #변수의 특성 중요도 순으로 정렬하기
    fi_temp.sort_values(by=['feature_importance'], ascending=False,inplace=True)
    fi_temp.reset_index(drop=True, inplace = True)

    if topn == 'all' :
        fi_df = fi_temp.copy()
    else :
        fi_df = fi_temp.iloc[:topn]

    #변수의 특성 중요도 그래프로 그리기
    if result_only == False :
        plt.figure(figsize=(10,20))
        sns.barplot(x='feature_importance', y='feature_name', data = fi_df)

        plt.xlabel('importance')
        plt.ylabel('feature name')
        plt.grid()

    return fi_df

### (2) 데이터 불러오기

* 주어진 데이터셋
    * data01_train.csv : 학습 및 검증용
* 세부 요구사항
    - 전체 데이터 'data01_train.csv' 를 불러와 'data' 이름으로 저장합니다.
        - data에서 변수 subject는 삭제합니다.
    - 데이터프레임에 대한 기본 정보를 확인합니다.( .head(), .shape 등)

#### 1) 데이터 로딩

In [7]:
train_data = pd.read_csv('data01_train.csv')

In [8]:
test_data = pd.read_csv('data01_test.csv')

#### 2) 기본 정보 조회

In [9]:
train_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5881 entries, 0 to 5880
Columns: 563 entries, tBodyAcc-mean()-X to Activity
dtypes: float64(561), int64(1), object(1)
memory usage: 25.3+ MB


In [10]:
train_data.head()

,tBodyAcc-mean()-X,tBodyAcc-mean()-Y,tBodyAcc-mean()-Z,tBodyAcc-std()-X,tBodyAcc-std()-Y,tBodyAcc-std()-Z,tBodyAcc-mad()-X,tBodyAcc-mad()-Y,tBodyAcc-mad()-Z,tBodyAcc-max()-X,...,fBodyBodyGyroJerkMag-kurtosis(),"angle(tBodyAccMean,gravity)","angle(tBodyAccJerkMean),gravityMean)","angle(tBodyGyroMean,gravityMean)","angle(tBodyGyroJerkMean,gravityMean)","angle(X,gravityMean)","angle(Y,gravityMean)","angle(Z,gravityMean)",subject,Activity
0,0.288508,-0.009196,-0.103362,-0.988986,-0.962797,-0.967422,-0.989000,-0.962596,-0.965650,-0.929747,...,-0.816696,-0.042494,-0.044218,0.307873,0.072790,-0.601120,0.331298,0.165163,21,STANDING
1,0.265757,-0.016576,-0.098163,-0.989551,-0.994636,-0.987435,-0.990189,-0.993870,-0.987558,-0.937337,...,-0.693515,-0.062899,0.388459,-0.765014,0.771524,0.345205,-0.769186,-0.147944,15,LAYING
2,0.278709,-0.014511,-0.108717,-0.997720,-0.981088,-0.994008,-0.997934,-0.982187,-0.995017,-0.942584,...,-0.829311,0.000265,-0.525022,-0.891875,0.021528,-0.833564,0.202434,-0.032755,11,STANDING
3,0.289795,-0.035536,-0.150354,-0.231727,-0.006412,-0.338117,-0.273557,0.014245,-0.347916,0.008288,...,-0.408956,-0.255125,0.612804,0.747381,-0.072944,-0.695819,0.287154,0.111388,17,WALKING
4,0.394807,0.034098,0.091229,0.088489,-0.106636,-0.388502,-0.010469,-0.109680,-0.346372,0.584131,...,-0.563437,-0.044344,-0.845268,-0.974650,-0.887846,-0.705029,0.264952,0.137758,17,WALKING_DOWNSTAIRS


In [11]:
train_data.describe()

,tBodyAcc-mean()-X,tBodyAcc-mean()-Y,tBodyAcc-mean()-Z,tBodyAcc-std()-X,tBodyAcc-std()-Y,tBodyAcc-std()-Z,tBodyAcc-mad()-X,tBodyAcc-mad()-Y,tBodyAcc-mad()-Z,tBodyAcc-max()-X,...,fBodyBodyGyroJerkMag-skewness(),fBodyBodyGyroJerkMag-kurtosis(),"angle(tBodyAccMean,gravity)","angle(tBodyAccJerkMean),gravityMean)","angle(tBodyGyroMean,gravityMean)","angle(tBodyGyroJerkMean,gravityMean)","angle(X,gravityMean)","angle(Y,gravityMean)","angle(Z,gravityMean)",subject
count,5881.000000,5881.000000,5881.000000,5881.000000,5881.000000,5881.000000,5881.000000,5881.000000,5881.000000,5881.000000,...,5881.000000,5881.000000,5881.000000,5881.000000,5881.000000,5881.000000,5881.000000,5881.000000,5881.000000,5881.000000
mean,0.274811,-0.017799,-0.109396,-0.603138,-0.509815,-0.604058,-0.628151,-0.525944,-0.605374,-0.465490,...,-0.305883,-0.623548,0.008524,-0.001185,0.009340,-0.007099,-0.491501,0.059299,-0.054594,17.381568
std,0.067614,0.039422,0.058373,0.448807,0.501815,0.417319,0.424345,0.485115,0.413043,0.544995,...,0.322808,0.310371,0.339730,0.447197,0.608190,0.476738,0.509069,0.297340,0.278479,8.938316
min,-0.503823,-0.684893,-1.000000,-1.000000,-0.999844,-0.999667,-1.000000,-0.999419,-1.000000,-1.000000,...,-0.979261,-0.999765,-0.976580,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-0.980143,1.000000
25%,0.262919,-0.024877,-0.121051,-0.992774,-0.977680,-0.980127,-0.993602,-0.977865,-0.980112,-0.936067,...,-0.541969,-0.845985,-0.122361,-0.294369,-0.481718,-0.373345,-0.811397,-0.018203,-0.141555,8.000000
50%,0.277154,-0.017221,-0.108781,-0.943933,-0.844575,-0.856352,-0.948501,-0.849266,-0.849896,-0.878729,...,-0.342923,-0.712677,0.010278,0.005146,0.011448,-0.000847,-0.709441,0.182893,0.003951,19.000000
75%,0.288526,-0.010920,-0.098163,-0.242130,-0.034499,-0.262690,-0.291138,-0.068857,-0.268539,-0.013690,...,-0.127371,-0.501158,0.154985,0.285030,0.499857,0.356236,-0.511330,0.248435,0.111932,26.000000
max,1.000000,1.000000,1.000000,1.000000,0.916238,1.000000,1.000000,0.967664,1.000000,1.000000,...,0.989538,0.956845,1.000000,1.000000,0.998702,0.996078,0.977344,0.478157,1.000000,30.000000


## **2. 데이터 전처리**

* 가변수화, 데이터 분할, NaN 확인 및 조치, 스케일링 등 필요한 전처리를 수행한다.


In [12]:
train_data.drop('subject', axis=1, inplace=True)

In [13]:
train_data.head(3)

,tBodyAcc-mean()-X,tBodyAcc-mean()-Y,tBodyAcc-mean()-Z,tBodyAcc-std()-X,tBodyAcc-std()-Y,tBodyAcc-std()-Z,tBodyAcc-mad()-X,tBodyAcc-mad()-Y,tBodyAcc-mad()-Z,tBodyAcc-max()-X,...,fBodyBodyGyroJerkMag-skewness(),fBodyBodyGyroJerkMag-kurtosis(),"angle(tBodyAccMean,gravity)","angle(tBodyAccJerkMean),gravityMean)","angle(tBodyGyroMean,gravityMean)","angle(tBodyGyroJerkMean,gravityMean)","angle(X,gravityMean)","angle(Y,gravityMean)","angle(Z,gravityMean)",Activity
0,0.288508,-0.009196,-0.103362,-0.988986,-0.962797,-0.967422,-0.989000,-0.962596,-0.965650,-0.929747,...,-0.487737,-0.816696,-0.042494,-0.044218,0.307873,0.072790,-0.601120,0.331298,0.165163,STANDING
1,0.265757,-0.016576,-0.098163,-0.989551,-0.994636,-0.987435,-0.990189,-0.993870,-0.987558,-0.937337,...,-0.237820,-0.693515,-0.062899,0.388459,-0.765014,0.771524,0.345205,-0.769186,-0.147944,LAYING
2,0.278709,-0.014511,-0.108717,-0.997720,-0.981088,-0.994008,-0.997934,-0.982187,-0.995017,-0.942584,...,-0.535287,-0.829311,0.000265,-0.525022,-0.891875,0.021528,-0.833564,0.202434,-0.032755,STANDING


In [14]:
sum(train_data.isna().sum())

0

### (1) 데이터 분할1 : x, y

* 세부 요구사항
    - x, y로 분할합니다.

In [15]:
dynamic_activity = (train_data['Activity']=='WALKING') | (train_data['Activity']=='WALKING_UPSTAIRS') | (train_data['Activity']=='WALKING_DOWNSTAIRS')
train_data = train_data[dynamic_activity]

In [16]:
train_data

,tBodyAcc-mean()-X,tBodyAcc-mean()-Y,tBodyAcc-mean()-Z,tBodyAcc-std()-X,tBodyAcc-std()-Y,tBodyAcc-std()-Z,tBodyAcc-mad()-X,tBodyAcc-mad()-Y,tBodyAcc-mad()-Z,tBodyAcc-max()-X,...,fBodyBodyGyroJerkMag-skewness(),fBodyBodyGyroJerkMag-kurtosis(),"angle(tBodyAccMean,gravity)","angle(tBodyAccJerkMean),gravityMean)","angle(tBodyGyroMean,gravityMean)","angle(tBodyGyroJerkMean,gravityMean)","angle(X,gravityMean)","angle(Y,gravityMean)","angle(Z,gravityMean)",Activity
3,0.289795,-0.035536,-0.150354,-0.231727,-0.006412,-0.338117,-0.273557,0.014245,-0.347916,0.008288,...,-0.004012,-0.408956,-0.255125,0.612804,0.747381,-0.072944,-0.695819,0.287154,0.111388,WALKING
4,0.394807,0.034098,0.091229,0.088489,-0.106636,-0.388502,-0.010469,-0.109680,-0.346372,0.584131,...,-0.157832,-0.563437,-0.044344,-0.845268,-0.974650,-0.887846,-0.705029,0.264952,0.137758,WALKING_DOWNSTAIRS
5,0.330708,0.007561,-0.061371,-0.215760,0.101075,0.072949,-0.269857,0.060060,0.101298,-0.019263,...,-0.599906,-0.887024,-0.030645,-0.852091,-0.500195,0.306091,-0.552729,0.253885,0.291256,WALKING_UPSTAIRS
6,0.121465,-0.031902,-0.005196,-0.152198,-0.113104,-0.239423,-0.202401,-0.164698,-0.247099,0.114668,...,-0.371055,-0.775779,0.445206,-0.003487,-0.940185,0.041387,-0.886603,0.173338,-0.005627,WALKING
12,0.303885,0.002768,-0.038613,-0.168656,0.190336,-0.140473,-0.205134,0.101144,-0.120572,-0.000818,...,-0.037750,-0.329728,-0.040030,0.257252,0.076091,-0.123425,-0.752882,0.266729,0.045692,WALKING
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5868,0.277709,-0.053919,-0.098746,-0.589970,-0.076626,-0.500837,-0.605474,-0.142798,-0.506696,-0.533485,...,0.029201,-0.268237,-0.076922,0.706620,-0.954969,-0.324779,-0.691663,0.153974,-0.196833,WALKING
5869,0.252496,-0.009773,-0.118293,-0.155168,-0.244513,-0.044524,-0.244367,-0.229509,-0.042199,0.291682,...,-0.475800,-0.752980,0.619246,-0.684483,0.924623,-0.353041,-0.775518,0.253218,0.013216,WALKING_UPSTAIRS
5873,0.264961,-0.034719,-0.088140,-0.368158,-0.074514,-0.229606,-0.394387,-0.082437,-0.243525,-0.261087,...,0.198624,-0.046157,0.037517,-0.903692,0.614933,-0.610918,-0.601604,0.335996,0.156518,WALKING
5877,0.191568,0.013328,-0.105174,-0.126969,-0.121729,-0.327480,-0.192523,-0.109923,-0.295286,0.078644,...,-0.622617,-0.879215,0.721718,0.623151,0.866858,-0.445660,-0.690278,0.303194,-0.044188,WALKING_UPSTAIRS


In [17]:
dynamic_activity = (test_data['Activity']=='WALKING') | (test_data['Activity']=='WALKING_UPSTAIRS') | (test_data['Activity']=='WALKING_DOWNSTAIRS')
test_data = test_data[dynamic_activity]

In [18]:
test_data.head()

,tBodyAcc-mean()-X,tBodyAcc-mean()-Y,tBodyAcc-mean()-Z,tBodyAcc-std()-X,tBodyAcc-std()-Y,tBodyAcc-std()-Z,tBodyAcc-mad()-X,tBodyAcc-mad()-Y,tBodyAcc-mad()-Z,tBodyAcc-max()-X,...,fBodyBodyGyroJerkMag-kurtosis(),"angle(tBodyAccMean,gravity)","angle(tBodyAccJerkMean),gravityMean)","angle(tBodyGyroMean,gravityMean)","angle(tBodyGyroJerkMean,gravityMean)","angle(X,gravityMean)","angle(Y,gravityMean)","angle(Z,gravityMean)",subject,Activity
2,0.305833,-0.041023,-0.087303,0.006880,0.182800,-0.237984,0.005642,0.028616,-0.236474,0.016311,...,0.394388,-0.362616,0.171069,0.576349,-0.688314,-0.743234,0.272186,0.053101,22,WALKING
5,0.336801,0.010528,-0.057819,-0.435988,-0.126317,0.037931,-0.446406,-0.244650,-0.034606,-0.288437,...,-0.351891,-0.174328,0.558749,0.664988,0.544573,-0.775712,0.187548,0.149686,23,WALKING
9,0.087902,-0.118993,-0.052787,-0.217817,-0.148752,0.079020,-0.294967,-0.165066,0.137718,0.070103,...,-0.842876,0.313940,0.577018,-0.968008,0.525585,-0.872144,0.186079,0.007999,30,WALKING_UPSTAIRS
16,0.267601,-0.032814,-0.113073,-0.290751,0.028228,-0.011504,-0.341515,0.043469,0.048429,-0.226421,...,-0.795217,-0.082706,0.206094,-0.917195,-0.365769,-0.616598,0.353675,0.087181,1,WALKING_UPSTAIRS
17,0.187920,-0.022493,-0.114320,-0.285203,-0.001438,0.163397,-0.339592,-0.134192,0.172202,0.093298,...,-0.806618,0.765867,-0.099255,0.815948,0.007978,-0.796053,0.199388,0.119815,23,WALKING


In [19]:
X_train = train_data.drop('Activity', axis=1)
y_train = train_data.loc[:, 'Activity']

In [20]:
X_test = test_data.drop(['subject', 'Activity'], axis=1)
y_test = test_data.loc[:, 'Activity']

In [21]:
X_test

,tBodyAcc-mean()-X,tBodyAcc-mean()-Y,tBodyAcc-mean()-Z,tBodyAcc-std()-X,tBodyAcc-std()-Y,tBodyAcc-std()-Z,tBodyAcc-mad()-X,tBodyAcc-mad()-Y,tBodyAcc-mad()-Z,tBodyAcc-max()-X,...,fBodyBodyGyroJerkMag-meanFreq(),fBodyBodyGyroJerkMag-skewness(),fBodyBodyGyroJerkMag-kurtosis(),"angle(tBodyAccMean,gravity)","angle(tBodyAccJerkMean),gravityMean)","angle(tBodyGyroMean,gravityMean)","angle(tBodyGyroJerkMean,gravityMean)","angle(X,gravityMean)","angle(Y,gravityMean)","angle(Z,gravityMean)"
2,0.305833,-0.041023,-0.087303,0.006880,0.182800,-0.237984,0.005642,0.028616,-0.236474,0.016311,...,0.468354,0.579587,0.394388,-0.362616,0.171069,0.576349,-0.688314,-0.743234,0.272186,0.053101
5,0.336801,0.010528,-0.057819,-0.435988,-0.126317,0.037931,-0.446406,-0.244650,-0.034606,-0.288437,...,0.157082,0.115015,-0.351891,-0.174328,0.558749,0.664988,0.544573,-0.775712,0.187548,0.149686
9,0.087902,-0.118993,-0.052787,-0.217817,-0.148752,0.079020,-0.294967,-0.165066,0.137718,0.070103,...,-0.133990,-0.535753,-0.842876,0.313940,0.577018,-0.968008,0.525585,-0.872144,0.186079,0.007999
16,0.267601,-0.032814,-0.113073,-0.290751,0.028228,-0.011504,-0.341515,0.043469,0.048429,-0.226421,...,0.483363,-0.475586,-0.795217,-0.082706,0.206094,-0.917195,-0.365769,-0.616598,0.353675,0.087181
17,0.187920,-0.022493,-0.114320,-0.285203,-0.001438,0.163397,-0.339592,-0.134192,0.172202,0.093298,...,0.278589,-0.381085,-0.806618,0.765867,-0.099255,0.815948,0.007978,-0.796053,0.199388,0.119815
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1459,0.330787,0.003088,-0.119242,-0.031168,0.153938,-0.347749,-0.102352,0.154328,-0.372959,0.189891,...,0.211394,-0.522999,-0.821387,-0.273972,0.164241,0.698665,0.793528,-0.553123,0.367753,0.166578
1462,0.305610,-0.016737,-0.030036,0.127349,0.144149,0.028455,0.061540,0.193795,0.013134,0.443721,...,0.430850,-0.112557,-0.463861,0.076715,0.332003,-0.928488,-0.360979,-0.640240,0.194490,0.255591
1463,0.333934,-0.015675,-0.125687,0.076394,-0.187450,0.144608,-0.023927,-0.231563,0.102194,0.557096,...,0.255564,-0.205663,-0.553521,-0.858843,-0.601687,0.893242,0.674606,-0.680206,0.165492,0.237212
1468,0.240402,0.006361,-0.121377,-0.045805,0.189930,0.332664,-0.114706,0.157771,0.195271,0.210139,...,0.187655,-0.213340,-0.615554,0.330378,-0.667635,0.806563,-0.850113,-0.639564,0.185363,0.260201


In [22]:
y_test

,Activity
2,WALKING
5,WALKING
9,WALKING_UPSTAIRS
16,WALKING_UPSTAIRS
17,WALKING
...,...
1459,WALKING_UPSTAIRS
1462,WALKING_DOWNSTAIRS
1463,WALKING_DOWNSTAIRS
1468,WALKING_DOWNSTAIRS


In [23]:
y_train.unique()

array(['WALKING', 'WALKING_DOWNSTAIRS', 'WALKING_UPSTAIRS'], dtype=object)

In [24]:
y_test.unique()

array(['WALKING', 'WALKING_UPSTAIRS', 'WALKING_DOWNSTAIRS'], dtype=object)

### (2) 스케일링(필요시)


* 세부 요구사항
    - 스케일링을 필요로 하는 알고리즘 사용을 위해서 코드 수행
    - min-max 방식 혹은 standard 방식 중 한가지 사용.

In [25]:
# 스케일링 필요 없다고 판단

### (3) 데이터분할2 : train, validation

* 세부 요구사항
    - train : val = 8 : 2 혹은 7 : 3
    - random_state 옵션을 사용하여 다른 모델과 비교를 위해 성능이 재현되도록 합니다.

In [26]:
# X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

## **3. 기본 모델링**



* 세부 요구사항
    - 최소 4개 이상의 알고리즘을 적용하여 모델링을 수행한다.
    - 각 알고리즘별로 전체 변수로 모델링, 상위 N개 변수를 선택하여 모델링을 수행하고 성능 비교를 한다.
    - (옵션) 알고리즘 중 1~2개에 대해서, 변수 중요도 상위 N개를 선정하여 모델링하고 타 모델과 성능을 비교.
        * 상위 N개를 선택하는 방법은, 변수를 하나씩 늘려가며 모델링 및 성능 검증을 수행하여 적절한 지점을 찾는 것이다.

In [27]:
acc_df = pd.DataFrame({'모델': [],
                       'default_정확도(교차검증)': [],
                       'default_정확도(테스트)': [],
                       'default_정확도(fi_교차검증)': [],
                       'default_정확도(fi_테스트)': []
                      })
acc_df

,모델,default_정확도(교차검증),default_정확도(테스트),default_정확도(fi_교차검증),default_정확도(fi_테스트)


In [28]:
kappa_df = pd.DataFrame({'모델': [],
                       'default_kappa(교차검증)': [],
                       'default_kappa(테스트)': [],
                      })
kappa_df

,모델,default_kappa(교차검증),default_kappa(테스트)


In [29]:
kappa_scorer = make_scorer(cohen_kappa_score)

### (1) 알고리즘1 : RandomForest

In [ ]:
rf_model = RandomForestClassifier(random_state=42)
scores = cross_validate(rf_model, X_train, y_train, cv=5, scoring={'accuracy': 'accuracy', 'kappa': kappa_scorer})
print(f"평균 정확도:", np.mean(scores['test_accuracy']))
print(f"평균 카파  :", np.mean(scores['test_kappa']))

평균 정확도: 0.9848892534864643
평균 카파  : 0.9772313246845291


In [ ]:
rf_model = RandomForestClassifier(random_state=42)

rf_model.fit(X_train, y_train)
y_pred = rf_model.predict(X_test)

print('accuracy :',accuracy_score(y_test, y_pred))
print('='*60)
print(confusion_matrix(y_test, y_pred))
print('='*60)
print(classification_report(y_test, y_pred))
print(f"Cohen's Kappa Score: {cohen_kappa_score(y_test, y_pred):.4f}")

accuracy : 0.9905956112852664
[[225   0   3]
 [  0 193   2]
 [  0   1 214]]
                    precision    recall  f1-score   support

           WALKING       1.00      0.99      0.99       228
WALKING_DOWNSTAIRS       0.99      0.99      0.99       195
  WALKING_UPSTAIRS       0.98      1.00      0.99       215

          accuracy                           0.99       638
         macro avg       0.99      0.99      0.99       638
      weighted avg       0.99      0.99      0.99       638

Cohen's Kappa Score: 0.9859


In [ ]:
joblib.dump(rf_model, 'rf_model_default.pkl')

['rf_model_default.pkl']

In [ ]:
new_row = pd.DataFrame([{'모델': 'RandomForest', 'default_정확도(교차검증)': np.mean(scores['test_accuracy']), 'default_정확도(테스트)': accuracy_score(y_test, y_pred)}])
acc_df = pd.concat([acc_df, new_row], ignore_index=True)

In [ ]:
new_row = pd.DataFrame([{'모델': 'RandomForest', 'default_kappa(교차검증)': np.mean(scores['test_kappa']), 'default_kappa(테스트)': cohen_kappa_score(y_test, y_pred)}])
kappa_df = pd.concat([kappa_df, new_row], ignore_index=True)

In [ ]:
acc_df

,모델,default_정확도(교차검증),default_정확도(테스트),default_정확도(fi_교차검증),default_정확도(fi_테스트)
0,RandomForest,0.984889,0.990596,NaN,NaN


In [ ]:
kappa_df

,모델,default_kappa(교차검증),default_kappa(테스트)
0,RandomForest,0.977231,0.985866


### (2) 알고리즘2 : LogisticRegression

In [ ]:
lr_model = LogisticRegression(max_iter=1000, random_state=42)
scores = cross_validate(lr_model, X_train, y_train, cv=5, scoring={'accuracy': 'accuracy', 'kappa': kappa_scorer})
print(f"평균 정확도:", np.mean(scores['test_accuracy']))
print(f"평균 카파  :", np.mean(scores['test_kappa']))

평균 정확도: 0.9954659913685486
평균 카파  : 0.9931653918943116


In [ ]:
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train, y_train)
y_pred = lr_model.predict(X_test)

print('accuracy :',accuracy_score(y_test, y_pred))
print('='*60)
print(confusion_matrix(y_test, y_pred))
print('='*60)
print(classification_report(y_test, y_pred))
print(f"Cohen's Kappa Score: {cohen_kappa_score(y_test, y_pred):.4f}")

accuracy : 0.9984326018808778
[[228   0   0]
 [  0 195   0]
 [  1   0 214]]
                    precision    recall  f1-score   support

           WALKING       1.00      1.00      1.00       228
WALKING_DOWNSTAIRS       1.00      1.00      1.00       195
  WALKING_UPSTAIRS       1.00      1.00      1.00       215

          accuracy                           1.00       638
         macro avg       1.00      1.00      1.00       638
      weighted avg       1.00      1.00      1.00       638

Cohen's Kappa Score: 0.9976


In [ ]:
joblib.dump(lr_model, 'lr_model_default.pkl')

['lr_model_default.pkl']

In [ ]:
new_row = pd.DataFrame([{'모델': 'LogisticRegression', 'default_정확도(교차검증)': np.mean(scores['test_accuracy']), 'default_정확도(테스트)': accuracy_score(y_test, y_pred)}])
acc_df = pd.concat([acc_df, new_row], ignore_index=True)

In [ ]:
new_row = pd.DataFrame([{'모델': 'LogisticRegression', 'default_kappa(교차검증)': np.mean(scores['test_kappa']), 'default_kappa(테스트)': cohen_kappa_score(y_test, y_pred)}])
kappa_df = pd.concat([kappa_df, new_row], ignore_index=True)

In [ ]:
acc_df

,모델,default_정확도(교차검증),default_정확도(테스트),default_정확도(fi_교차검증),default_정확도(fi_테스트)
0,RandomForest,0.984889,0.990596,NaN,NaN
1,LogisticRegression,0.995466,0.998433,NaN,NaN


In [ ]:
kappa_df

,모델,default_kappa(교차검증),default_kappa(테스트)
0,RandomForest,0.977231,0.985866
1,LogisticRegression,0.993165,0.997644


### (3) 알고리즘3 : SVM

In [ ]:
svm_model = SVC(random_state=42)
scores = cross_validate(svm_model, X_train, y_train, cv=5, scoring={'accuracy': 'accuracy', 'kappa': kappa_scorer})
print(f"평균 정확도:", np.mean(scores['test_accuracy']))
print(f"평균 카파  :", np.mean(scores['test_kappa']))

평균 정확도: 0.9935777722295537
평균 카파  : 0.9903194948544796


In [ ]:
svm_model = SVC(random_state=42)
svm_model.fit(X_train, y_train)

y_pred = svm_model.predict(X_test)

print('accuracy :',accuracy_score(y_test, y_pred))
print('='*60)
print(confusion_matrix(y_test, y_pred))
print('='*60)
print(classification_report(y_test, y_pred))
print(f"Cohen's Kappa Score: {cohen_kappa_score(y_test, y_pred):.4f}")

accuracy : 0.9984326018808778
[[228   0   0]
 [  0 195   0]
 [  1   0 214]]
                    precision    recall  f1-score   support

           WALKING       1.00      1.00      1.00       228
WALKING_DOWNSTAIRS       1.00      1.00      1.00       195
  WALKING_UPSTAIRS       1.00      1.00      1.00       215

          accuracy                           1.00       638
         macro avg       1.00      1.00      1.00       638
      weighted avg       1.00      1.00      1.00       638

Cohen's Kappa Score: 0.9976


In [ ]:
joblib.dump(svm_model, 'svm_model_default.pkl')

['svm_model_default.pkl']

In [ ]:
new_row = pd.DataFrame([{'모델': 'SVM', 'default_정확도(교차검증)': np.mean(scores['test_accuracy']), 'default_정확도(테스트)': accuracy_score(y_test, y_pred)}])
acc_df = pd.concat([acc_df, new_row], ignore_index=True)

In [ ]:
new_row = pd.DataFrame([{'모델': 'SVM', 'default_kappa(교차검증)': np.mean(scores['test_kappa']), 'default_kappa(테스트)': cohen_kappa_score(y_test, y_pred)}])
kappa_df = pd.concat([kappa_df, new_row], ignore_index=True)

In [ ]:
acc_df

,모델,default_정확도(교차검증),default_정확도(테스트),default_정확도(fi_교차검증),default_정확도(fi_테스트)
0,RandomForest,0.984889,0.990596,NaN,NaN
1,LogisticRegression,0.995466,0.998433,NaN,NaN
2,SVM,0.993578,0.998433,NaN,NaN


In [ ]:
kappa_df

,모델,default_kappa(교차검증),default_kappa(테스트)
0,RandomForest,0.977231,0.985866
1,LogisticRegression,0.993165,0.997644
2,SVM,0.990319,0.997644


### (4) 알고리즘4 : DecisionTreeClassifier

In [ ]:
dt_model = DecisionTreeClassifier(random_state=42)
scores = cross_validate(dt_model, X_train, y_train, cv=5, scoring={'accuracy': 'accuracy', 'kappa': kappa_scorer})
print(f"평균 정확도:", np.mean(scores['test_accuracy']))
print(f"평균 카파  :", np.mean(scores['test_kappa']))

평균 정확도: 0.9297328530156579
평균 카파  : 0.8940928212182244


In [ ]:
dt_model = DecisionTreeClassifier(random_state=42)
dt_model.fit(X_train, y_train)

y_pred = dt_model.predict(X_test)

print('accuracy :',accuracy_score(y_test, y_pred))
print('='*60)
print(confusion_matrix(y_test, y_pred))
print('='*60)
print(classification_report(y_test, y_pred))
print(f"Cohen's Kappa Score: {cohen_kappa_score(y_test, y_pred):.4f}")

accuracy : 0.9420062695924765
[[214   2  12]
 [  1 188   6]
 [  6  10 199]]
                    precision    recall  f1-score   support

           WALKING       0.97      0.94      0.95       228
WALKING_DOWNSTAIRS       0.94      0.96      0.95       195
  WALKING_UPSTAIRS       0.92      0.93      0.92       215

          accuracy                           0.94       638
         macro avg       0.94      0.94      0.94       638
      weighted avg       0.94      0.94      0.94       638

Cohen's Kappa Score: 0.9129


In [ ]:
joblib.dump(dt_model, 'dt_model_default.pkl')

['dt_model_default.pkl']

In [ ]:
new_row = pd.DataFrame([{'모델': 'DecisionTreeClassifier', 'default_정확도(교차검증)': np.mean(scores['test_accuracy']), 'default_정확도(테스트)': accuracy_score(y_test, y_pred)}])
acc_df = pd.concat([acc_df, new_row], ignore_index=True)

In [ ]:
new_row = pd.DataFrame([{'모델': 'DecisionTreeClassifier', 'default_kappa(교차검증)': np.mean(scores['test_kappa']), 'default_kappa(테스트)': cohen_kappa_score(y_test, y_pred)}])
kappa_df = pd.concat([kappa_df, new_row], ignore_index=True)

In [ ]:
acc_df

,모델,default_정확도(교차검증),default_정확도(테스트),default_정확도(fi_교차검증),default_정확도(fi_테스트)
0,RandomForest,0.984889,0.990596,NaN,NaN
1,LogisticRegression,0.995466,0.998433,NaN,NaN
2,SVM,0.993578,0.998433,NaN,NaN
3,DecisionTreeClassifier,0.929733,0.942006,NaN,NaN


In [ ]:
kappa_df

,모델,default_kappa(교차검증),default_kappa(테스트)
0,RandomForest,0.977231,0.985866
1,LogisticRegression,0.993165,0.997644
2,SVM,0.990319,0.997644
3,DecisionTreeClassifier,0.894093,0.912893


In [ ]:
acc_df.to_csv('모델 정확도(디폴트).csv', index=False, encoding='utf-8-sig')

In [ ]:
kappa_df.to_csv('모델 kappa(디폴트).csv', index=False, encoding='utf-8-sig')

## ================일단 여기까지=====================

### (5) 알고리즘5 : XGBoost

In [ ]:
# le = LabelEncoder()
# y_xgb = le.fit_transform(y)

In [ ]:
# X_train_xgb, X_val_xgb, y_train_xgb, y_val_xgb = train_test_split(X, y_xgb, test_size=0.2, random_state=42)

In [ ]:
# XGB_model = XGBClassifier(random_state=42)
# XGB_model.fit(X_train_xgb, y_train_xgb)

# y_pred = XGB_model.predict(X_val_xgb)
# y_pred = le.inverse_transform(y_pred)

# y_val = le.inverse_transform(y_val_xgb)

# print('accuracy :',accuracy_score(y_val, y_pred))
# print('='*60)
# print(confusion_matrix(y_val, y_pred))
# print('='*60)
# print(classification_report(y_val, y_pred))

In [ ]:
# new_row = pd.DataFrame([{'model': 'XGBoost', 'accuracy': accuracy_score(y_val, y_pred)}])
# acc_df = pd.concat([acc_df, new_row], ignore_index=True)

In [ ]:
# acc_df

### (6) 알고리즘6 : LightGBM

In [ ]:
# X_lgb = X.copy()
# X_lgb.columns = [f'feature_{i}' for i in range(X.shape[1])]

In [ ]:
# le = LabelEncoder()
# y_lgb = le.fit_transform(y)

In [ ]:
# X_train_lgb, X_val_lgb, y_train_lgb, y_val_lgb = train_test_split(X_lgb, y_lgb, test_size=0.2, random_state=42)

In [ ]:
# LGB_model = lgb.LGBMClassifier(random_state=42)
# LGB_model.fit(X_train_lgb, y_train_lgb)  # 경고문이 나옴. 너무 많은 특성 중 쓸모 없는 게 많을 때 이런 경고가 뜰 수 있음. 특성 중요도로 피처 수를 조절할 필요가 있음.

# y_pred = LGB_model.predict(X_val_lgb)
# y_pred = le.inverse_transform(y_pred)

# y_val= le.inverse_transform(y_val_lgb)

# print('accuracy :',accuracy_score(y_val, y_pred))
# print('='*60)
# print(confusion_matrix(y_val, y_pred))
# print('='*60)
# print(classification_report(y_val, y_pred))

### (7) 알고리즘7 : ANN

In [ ]:
# X = data.drop('Activity', axis=1)
# y = data.loc[:, 'Activity'].values

In [ ]:
# from sklearn.preprocessing import LabelEncoder
# le = LabelEncoder()
# y_encoded = le.fit_transform(y)

In [ ]:
# x_train, x_val, y_train, y_val = train_test_split(X, y_encoded, test_size=0.2, random_state=42)

In [ ]:
# from tensorflow.keras.models import Sequential
# from tensorflow.keras.layers import Input, Dense, Dropout
# model = Sequential()

In [ ]:
# model.add(Input(shape=(561,)))
# model.add(Dense(283, activation='relu'))
# model.add(Dropout(0.3))
# model.add(Dense(units=283, activation='relu'))
# model.add(Dropout(0.3))
# model.add(Dense(units=283, activation='relu'))
# model.add(Dropout(0.3))
# model.add(Dense(units=6, activation='softmax'))

In [ ]:
# model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

In [ ]:
# model.fit(x_train, y_train, batch_size = 32, epochs = 50)

In [ ]:
# y_pred = model.predict(x_val)
# y_pred = np.argmax(y_pred, axis=1)

In [ ]:
# y_val = le.inverse_transform(y_val)
# y_pred = le.inverse_transform(y_pred)
# print('accuracy :',accuracy_score(y_val, y_pred))
# print('='*60)
# print(confusion_matrix(y_val, y_pred))
# print('='*60)
# print(classification_report(y_val, y_pred))

## 특성 중요도(랜덤 포레스트에서 추출)

In [ ]:
acc_df = pd.read_csv('모델 정확도(디폴트).csv')
acc_df

,모델,default_정확도(교차검증),default_정확도(테스트),default_정확도(fi_교차검증),default_정확도(fi_테스트)
0,RandomForest,0.984889,0.990596,NaN,NaN
1,LogisticRegression,0.995466,0.998433,NaN,NaN
2,SVM,0.993578,0.998433,NaN,NaN
3,DecisionTreeClassifier,0.929733,0.942006,NaN,NaN


In [ ]:
rf_model = joblib.load('rf_model_default.pkl')

In [ ]:
RFmodel_fi_df = plot_feature_importance(rf_model.feature_importances_, list(X_train), True)
RFmodel_fi_df

,feature_name,feature_importance
0,"tGravityAcc-arCoeff()-Z,2",0.035384
1,fBodyAccMag-mad(),0.028719
2,fBodyAccMag-energy(),0.027388
3,tBodyAccMag-std(),0.026653
4,"tGravityAcc-arCoeff()-Z,3",0.024331
...,...,...
556,"tBodyGyro-arCoeff()-Y,3",0.000045
557,tGravityAcc-entropy()-Y,0.000044
558,tBodyAccJerk-mean()-Y,0.000044
559,"fBodyAccJerk-bandsEnergy()-57,64.2",0.000043


In [ ]:
# 기존 변수 개수
len(RFmodel_fi_df)

561

In [ ]:
RFmodel_fi_df['cum_importance'] = RFmodel_fi_df['feature_importance'].cumsum() / RFmodel_fi_df['feature_importance'].sum()

In [ ]:
# 누적 중요도 기준 95%까지 필요한 변수 개수
selected_features = RFmodel_fi_df[RFmodel_fi_df['cum_importance'] <= 0.95]['feature_importance']
print(f"누적 중요도 95%까지 선택된 변수 개수: {len(selected_features)}")

누적 중요도 95%까지 선택된 변수 개수: 299


In [ ]:
selected_cum_importance = RFmodel_fi_df.loc[:len(selected_features), 'feature_importance'].sum()
print(f"상위 {len(selected_features)+1}개 변수의 누적 중요도: {selected_cum_importance:.4f}")

상위 300개 변수의 누적 중요도: 0.9503


In [ ]:
X_train_fi = train_data[RFmodel_fi_df['feature_name'][:len(selected_features)+1]]
y_train_fi = train_data.loc[:, 'Activity']

In [ ]:
X_test_fi = test_data[RFmodel_fi_df['feature_name'][:len(selected_features)+1]]

### 랜덤 포레스트

In [ ]:
rf_model = RandomForestClassifier(random_state=42)
scores = cross_validate(rf_model, X_train_fi, y_train_fi, cv=5, scoring={'accuracy': 'accuracy', 'kappa': kappa_scorer})
print(f"평균 정확도:", np.mean(scores['test_accuracy']))
print(f"평균 카파  :", np.mean(scores['test_kappa']))

평균 정확도: 0.9871569711452723
평균 카파  : 0.980643838223019


In [ ]:
random_forest_model_fi = RandomForestClassifier(random_state=42)

random_forest_model_fi.fit(X_train_fi, y_train_fi)
y_pred = random_forest_model_fi.predict(X_test_fi)

print('accuracy :',accuracy_score(y_test, y_pred))
print('='*60)
print(confusion_matrix(y_test, y_pred))
print('='*60)
print(classification_report(y_test, y_pred))
print(f"Cohen's Kappa Score: {cohen_kappa_score(y_test, y_pred):.4f}")

accuracy : 0.9890282131661442
[[224   0   4]
 [  0 193   2]
 [  0   1 214]]
                    precision    recall  f1-score   support

           WALKING       1.00      0.98      0.99       228
WALKING_DOWNSTAIRS       0.99      0.99      0.99       195
  WALKING_UPSTAIRS       0.97      1.00      0.98       215

          accuracy                           0.99       638
         macro avg       0.99      0.99      0.99       638
      weighted avg       0.99      0.99      0.99       638

Cohen's Kappa Score: 0.9835


In [ ]:
joblib.dump(rf_model, 'rf_model_default_fi.pkl')

['rf_model_default_fi.pkl']

In [ ]:
acc_df

,모델,default_정확도(교차검증),default_정확도(테스트),default_정확도(fi_교차검증),default_정확도(fi_테스트)
0,RandomForest,0.984889,0.990596,NaN,NaN
1,LogisticRegression,0.995466,0.998433,NaN,NaN
2,SVM,0.993578,0.998433,NaN,NaN
3,DecisionTreeClassifier,0.929733,0.942006,NaN,NaN


In [ ]:
acc_df.loc[0, 'default_정확도(fi_교차검증)'] = np.mean(scores['test_accuracy'])
acc_df.loc[0, 'default_정확도(fi_테스트)'] = accuracy_score(y_test, y_pred)

In [ ]:
kappa_df.loc[0, 'default_kappa(fi_교차검증)'] = np.mean(scores['test_kappa'])
kappa_df.loc[0, 'default_kappa(fi_테스트)'] = cohen_kappa_score(y_test, y_pred)

In [ ]:
acc_df

,모델,default_정확도(교차검증),default_정확도(테스트),default_정확도(fi_교차검증),default_정확도(fi_테스트)
0,RandomForest,0.984889,0.990596,0.987157,0.989028
1,LogisticRegression,0.995466,0.998433,NaN,NaN
2,SVM,0.993578,0.998433,NaN,NaN
3,DecisionTreeClassifier,0.929733,0.942006,NaN,NaN


In [ ]:
kappa_df

,모델,default_kappa(교차검증),default_kappa(테스트),default_kappa(fi_교차검증),default_kappa(fi_테스트)
0,RandomForest,0.977231,0.985866,0.980644,0.983511
1,LogisticRegression,0.993165,0.997644,NaN,NaN
2,SVM,0.990319,0.997644,NaN,NaN
3,DecisionTreeClassifier,0.894093,0.912893,NaN,NaN


### 로지스틱회귀

In [ ]:
# 랜덤 포레스트에서 추출한 중요도 상위 특성 활용
lr_model = LogisticRegression(max_iter=1000, random_state=42)
scores = cross_validate(lr_model, X_train_fi, y_train_fi, cv=5, scoring={'accuracy': 'accuracy', 'kappa': kappa_scorer})
print(f"평균 정확도:", np.mean(scores['test_accuracy']))
print(f"평균 카파  :", np.mean(scores['test_kappa']))

평균 정확도: 0.9958447765452793
평균 카파  : 0.9937363921612056


In [ ]:
logistic_model = LogisticRegression(max_iter=1000, random_state=42)
logistic_model.fit(X_train_fi, y_train_fi)
y_pred = logistic_model.predict(X_test_fi)

print('accuracy :',accuracy_score(y_test, y_pred))
print('='*60)
print(confusion_matrix(y_test, y_pred))
print('='*60)
print(classification_report(y_test, y_pred))
print(f"Cohen's Kappa Score: {cohen_kappa_score(y_test, y_pred):.4f}")

accuracy : 0.9984326018808778
[[228   0   0]
 [  0 195   0]
 [  1   0 214]]
                    precision    recall  f1-score   support

           WALKING       1.00      1.00      1.00       228
WALKING_DOWNSTAIRS       1.00      1.00      1.00       195
  WALKING_UPSTAIRS       1.00      1.00      1.00       215

          accuracy                           1.00       638
         macro avg       1.00      1.00      1.00       638
      weighted avg       1.00      1.00      1.00       638

Cohen's Kappa Score: 0.9976


In [ ]:
joblib.dump(lr_model, 'lr_model_default_fi.pkl')

['lr_model_default_fi.pkl']

In [ ]:
acc_df.loc[1, 'default_정확도(fi_교차검증)'] = np.mean(scores['test_accuracy'])
acc_df.loc[1, 'default_정확도(fi_테스트)'] = accuracy_score(y_test, y_pred)

In [ ]:
kappa_df.loc[1, 'default_kappa(fi_교차검증)'] = np.mean(scores['test_kappa'])
kappa_df.loc[1, 'default_kappa(fi_테스트)'] = cohen_kappa_score(y_test, y_pred)

In [ ]:
acc_df

,모델,default_정확도(교차검증),default_정확도(테스트),default_정확도(fi_교차검증),default_정확도(fi_테스트)
0,RandomForest,0.984889,0.990596,0.987157,0.989028
1,LogisticRegression,0.995466,0.998433,0.995845,0.998433
2,SVM,0.993578,0.998433,NaN,NaN
3,DecisionTreeClassifier,0.929733,0.942006,NaN,NaN


In [ ]:
kappa_df

,모델,default_kappa(교차검증),default_kappa(테스트),default_kappa(fi_교차검증),default_kappa(fi_테스트)
0,RandomForest,0.977231,0.985866,0.980644,0.983511
1,LogisticRegression,0.993165,0.997644,0.993736,0.997644
2,SVM,0.990319,0.997644,NaN,NaN
3,DecisionTreeClassifier,0.894093,0.912893,NaN,NaN


### SVM

In [ ]:
svm_model = SVC(random_state=42)
scores = cross_validate(svm_model, X_train_fi, y_train_fi, cv=5, scoring={'accuracy': 'accuracy', 'kappa': kappa_scorer})
print(f"평균 정확도:", np.mean(scores['test_accuracy']))
print(f"평균 카파  :", np.mean(scores['test_kappa']))

평균 정확도: 0.995466704711631
평균 카파  : 0.9931664307668153


In [ ]:
svm_model = SVC(random_state=42)
svm_model.fit(X_train_fi, y_train_fi)

y_pred = svm_model.predict(X_test_fi)

print('accuracy :',accuracy_score(y_test, y_pred))
print('='*60)
print(confusion_matrix(y_test, y_pred))
print('='*60)
print(classification_report(y_test, y_pred))
print(f"Cohen's Kappa Score: {cohen_kappa_score(y_test, y_pred):.4f}")

accuracy : 0.9984326018808778
[[228   0   0]
 [  0 195   0]
 [  1   0 214]]
                    precision    recall  f1-score   support

           WALKING       1.00      1.00      1.00       228
WALKING_DOWNSTAIRS       1.00      1.00      1.00       195
  WALKING_UPSTAIRS       1.00      1.00      1.00       215

          accuracy                           1.00       638
         macro avg       1.00      1.00      1.00       638
      weighted avg       1.00      1.00      1.00       638

Cohen's Kappa Score: 0.9976


In [ ]:
joblib.dump(svm_model, 'svm_model_default_fi.pkl')

['svm_model_default_fi.pkl']

In [ ]:
acc_df.loc[2, 'default_정확도(fi_교차검증)'] = np.mean(scores['test_accuracy'])
acc_df.loc[2, 'default_정확도(fi_테스트)'] = accuracy_score(y_test, y_pred)

In [ ]:
kappa_df.loc[2, 'default_kappa(fi_교차검증)'] = np.mean(scores['test_kappa'])
kappa_df.loc[2, 'default_kappa(fi_테스트)'] = cohen_kappa_score(y_test, y_pred)

In [ ]:
acc_df

,모델,default_정확도(교차검증),default_정확도(테스트),default_정확도(fi_교차검증),default_정확도(fi_테스트)
0,RandomForest,0.984889,0.990596,0.987157,0.989028
1,LogisticRegression,0.995466,0.998433,0.995845,0.998433
2,SVM,0.993578,0.998433,0.995467,0.998433
3,DecisionTreeClassifier,0.929733,0.942006,NaN,NaN


In [ ]:
kappa_df

,모델,default_kappa(교차검증),default_kappa(테스트),default_kappa(fi_교차검증),default_kappa(fi_테스트)
0,RandomForest,0.977231,0.985866,0.980644,0.983511
1,LogisticRegression,0.993165,0.997644,0.993736,0.997644
2,SVM,0.990319,0.997644,0.993166,0.997644
3,DecisionTreeClassifier,0.894093,0.912893,NaN,NaN


### DecisionTreeClassifier

In [ ]:
dt_model = DecisionTreeClassifier(random_state=42)
scores = cross_validate(dt_model, X_train_fi, y_train_fi, cv=5, scoring={'accuracy': 'accuracy', 'kappa': kappa_scorer})
print(f"평균 정확도:", np.mean(scores['test_accuracy']))
print(f"평균 카파  :", np.mean(scores['test_kappa']))

평균 정확도: 0.933512144665977
평균 카파  : 0.8998149890906214


In [ ]:
dt_model = DecisionTreeClassifier(random_state=42)
dt_model.fit(X_train_fi, y_train_fi)

y_pred = dt_model.predict(X_test_fi)

print('accuracy :',accuracy_score(y_test, y_pred))
print('='*60)
print(confusion_matrix(y_test, y_pred))
print('='*60)
print(classification_report(y_test, y_pred))
print(f"Cohen's Kappa Score: {cohen_kappa_score(y_test, y_pred):.4f}")

accuracy : 0.9420062695924765
[[214   2  12]
 [  2 186   7]
 [  7   7 201]]
                    precision    recall  f1-score   support

           WALKING       0.96      0.94      0.95       228
WALKING_DOWNSTAIRS       0.95      0.95      0.95       195
  WALKING_UPSTAIRS       0.91      0.93      0.92       215

          accuracy                           0.94       638
         macro avg       0.94      0.94      0.94       638
      weighted avg       0.94      0.94      0.94       638

Cohen's Kappa Score: 0.9129


In [ ]:
joblib.dump(dt_model, 'dt_model_default_fi.pkl')

['dt_model_default_fi.pkl']

In [ ]:
acc_df.loc[3, 'default_정확도(fi_교차검증)'] = np.mean(scores['test_accuracy'])
acc_df.loc[3, 'default_정확도(fi_테스트)'] = accuracy_score(y_test, y_pred)

In [ ]:
kappa_df.loc[3, 'default_kappa(fi_교차검증)'] = np.mean(scores['test_kappa'])
kappa_df.loc[3, 'default_kappa(fi_테스트)'] = cohen_kappa_score(y_test, y_pred)

In [ ]:
acc_df

,모델,default_정확도(교차검증),default_정확도(테스트),default_정확도(fi_교차검증),default_정확도(fi_테스트)
0,RandomForest,0.984889,0.990596,0.987157,0.989028
1,LogisticRegression,0.995466,0.998433,0.995845,0.998433
2,SVM,0.993578,0.998433,0.995467,0.998433
3,DecisionTreeClassifier,0.929733,0.942006,0.933512,0.942006


In [ ]:
kappa_df

,모델,default_kappa(교차검증),default_kappa(테스트),default_kappa(fi_교차검증),default_kappa(fi_테스트)
0,RandomForest,0.977231,0.985866,0.980644,0.983511
1,LogisticRegression,0.993165,0.997644,0.993736,0.997644
2,SVM,0.990319,0.997644,0.993166,0.997644
3,DecisionTreeClassifier,0.894093,0.912893,0.899815,0.912853


In [ ]:
acc_df.to_csv('모델 정확도(디폴트_fi).csv', index=False, encoding='utf-8-sig')

In [ ]:
kappa_df.to_csv('모델 kappa(디폴트_fi).csv', index=False, encoding='utf-8-sig')

##방법2. PCA 활용

In [ ]:
acc_df = pd.read_csv('모델 정확도(디폴트_fi).csv')
acc_df

,모델,default_정확도(교차검증),default_정확도(테스트),default_정확도(fi_교차검증),default_정확도(fi_테스트)
0,RandomForest,0.984889,0.990596,0.987157,0.989028
1,LogisticRegression,0.995466,0.998433,0.995845,0.998433
2,SVM,0.993578,0.998433,0.995467,0.998433
3,DecisionTreeClassifier,0.929733,0.942006,0.933512,0.942006


In [46]:
from sklearn.decomposition import PCA
pca = PCA(n_components=0.95)
X_train_pca = pca.fit_transform(X_train)

print("선택된 주성분 수:", pca.n_components_)
print("누적 설명력:", pca.explained_variance_ratio_.cumsum()[-1])

선택된 주성분 수: 100
누적 설명력: 0.9504960804927014


In [40]:
X_test_pca = pca.transform(X_test)

### 랜덤포레스트

In [47]:
rf_model = RandomForestClassifier(random_state=42)
scores = cross_validate(rf_model, X_train_pca, y_train, cv=5, scoring={'accuracy': 'accuracy', 'kappa': kappa_scorer})
print(f"평균 정확도:", np.mean(scores['test_accuracy']))
print(f"평균 카파  :", np.mean(scores['test_kappa']))

평균 정확도: 0.9618418518386418
평균 카파  : 0.9424350974078347


In [ ]:
rf_model = RandomForestClassifier(random_state=42)

rf_model.fit(X_train_pca, y_train)
y_pred = rf_model.predict(X_test_pca)

print('accuracy :',accuracy_score(y_test, y_pred))
print('='*60)
print(confusion_matrix(y_test, y_pred))
print('='*60)
print(classification_report(y_test, y_pred))
print(f"Cohen's Kappa Score: {cohen_kappa_score(y_test, y_pred):.4f}")

accuracy : 0.9702194357366771
[[226   1   1]
 [  2 190   3]
 [ 10   2 203]]
                    precision    recall  f1-score   support

           WALKING       0.95      0.99      0.97       228
WALKING_DOWNSTAIRS       0.98      0.97      0.98       195
  WALKING_UPSTAIRS       0.98      0.94      0.96       215

          accuracy                           0.97       638
         macro avg       0.97      0.97      0.97       638
      weighted avg       0.97      0.97      0.97       638

Cohen's Kappa Score: 0.9552


In [ ]:
acc_df.loc[0, 'default_정확도(pca_교차검증)'] = np.mean(scores['test_accuracy'])
acc_df.loc[0, 'default_정확도(pca_테스트)'] = accuracy_score(y_test, y_pred)

In [ ]:
kappa_df.loc[0, 'default_kappa(pca_교차검증)'] = np.mean(scores['test_kappa'])
kappa_df.loc[0, 'default_kappa(pca_테스트)'] = cohen_kappa_score(y_test, y_pred)

In [ ]:
acc_df

,모델,default_정확도(교차검증),default_정확도(테스트),default_정확도(fi_교차검증),default_정확도(fi_테스트),default_정확도(pca_교차검증),default_정확도(pca_테스트)
0,RandomForest,0.984889,0.990596,0.987157,0.989028,0.961842,0.970219
1,LogisticRegression,0.995466,0.998433,0.995845,0.998433,NaN,NaN
2,SVM,0.993578,0.998433,0.995467,0.998433,NaN,NaN
3,DecisionTreeClassifier,0.929733,0.942006,0.933512,0.942006,NaN,NaN


In [ ]:
kappa_df

,모델,default_kappa(교차검증),default_kappa(테스트),default_kappa(fi_교차검증),default_kappa(fi_테스트),default_kappa(pca_교차검증),default_kappa(pca_테스트)
0,RandomForest,0.977231,0.985866,0.980644,0.983511,0.942435,0.95521
1,LogisticRegression,0.993165,0.997644,0.993736,0.997644,NaN,NaN
2,SVM,0.990319,0.997644,0.993166,0.997644,NaN,NaN
3,DecisionTreeClassifier,0.894093,0.912893,0.899815,0.912853,NaN,NaN


### 로지스틱회귀

In [48]:
lr_model = LogisticRegression(max_iter=1000, random_state=42) # 최적점에 도달 못하여 max_iter=1000로 설정
scores = cross_validate(lr_model, X_train_pca, y_train, cv=5, scoring={'accuracy': 'accuracy', 'kappa': kappa_scorer})
print(f"평균 정확도:", np.mean(scores['test_accuracy']))
print(f"평균 카파  :", np.mean(scores['test_kappa']))

평균 정확도: 0.9939551307201198
평균 카파  : 0.9908886940237036


In [ ]:
lr_model = LogisticRegression(max_iter=1000, random_state=42) # 최적점에 도달 못하여 max_iter=1000로 설정
lr_model.fit(X_train_pca, y_train)
y_pred = lr_model.predict(X_test_pca)

print('accuracy :',accuracy_score(y_test, y_pred))
print('='*60)
print(confusion_matrix(y_test, y_pred))
print('='*60)
print(classification_report(y_test, y_pred))
print(f"Cohen's Kappa Score: {cohen_kappa_score(y_test, y_pred):.4f}")

accuracy : 0.9968652037617555
[[227   0   1]
 [  0 195   0]
 [  1   0 214]]
                    precision    recall  f1-score   support

           WALKING       1.00      1.00      1.00       228
WALKING_DOWNSTAIRS       1.00      1.00      1.00       195
  WALKING_UPSTAIRS       1.00      1.00      1.00       215

          accuracy                           1.00       638
         macro avg       1.00      1.00      1.00       638
      weighted avg       1.00      1.00      1.00       638

Cohen's Kappa Score: 0.9953


In [ ]:
acc_df.loc[1, 'default_정확도(pca_교차검증)'] = np.mean(scores['test_accuracy'])
acc_df.loc[1, 'default_정확도(pca_테스트)'] = accuracy_score(y_test, y_pred)

In [ ]:
kappa_df.loc[1, 'default_kappa(pca_교차검증)'] = np.mean(scores['test_kappa'])
kappa_df.loc[1, 'default_kappa(pca_테스트)'] = cohen_kappa_score(y_test, y_pred)

In [ ]:
acc_df

,모델,default_정확도(교차검증),default_정확도(테스트),default_정확도(fi_교차검증),default_정확도(fi_테스트),default_정확도(pca_교차검증),default_정확도(pca_테스트)
0,RandomForest,0.984889,0.990596,0.987157,0.989028,0.961842,0.970219
1,LogisticRegression,0.995466,0.998433,0.995845,0.998433,0.993955,0.996865
2,SVM,0.993578,0.998433,0.995467,0.998433,NaN,NaN
3,DecisionTreeClassifier,0.929733,0.942006,0.933512,0.942006,NaN,NaN


In [ ]:
kappa_df

,모델,default_kappa(교차검증),default_kappa(테스트),default_kappa(fi_교차검증),default_kappa(fi_테스트),default_kappa(pca_교차검증),default_kappa(pca_테스트)
0,RandomForest,0.977231,0.985866,0.980644,0.983511,0.942435,0.955210
1,LogisticRegression,0.993165,0.997644,0.993736,0.997644,0.990889,0.995288
2,SVM,0.990319,0.997644,0.993166,0.997644,NaN,NaN
3,DecisionTreeClassifier,0.894093,0.912893,0.899815,0.912853,NaN,NaN


### SVM

In [ ]:
svm_model = SVC(random_state=42)
scores = cross_validate(svm_model, X_train_pca, y_train, cv=5, scoring={'accuracy': 'accuracy', 'kappa': kappa_scorer})
print(f"평균 정확도:", np.mean(scores['test_accuracy']))
print(f"평균 카파  :", np.mean(scores['test_kappa']))

평균 정확도: 0.9984891393515711
평균 카파  : 0.9977229316219093


In [ ]:
svm_model = SVC(random_state=42)
svm_model.fit(X_train_pca, y_train)

y_pred = svm_model.predict(X_test_pca)

print('accuracy :',accuracy_score(y_test, y_pred))
print('='*60)
print(confusion_matrix(y_test, y_pred))
print('='*60)
print(classification_report(y_test, y_pred))
print(f"Cohen's Kappa Score: {cohen_kappa_score(y_test, y_pred):.4f}")

accuracy : 1.0
[[228   0   0]
 [  0 195   0]
 [  0   0 215]]
                    precision    recall  f1-score   support

           WALKING       1.00      1.00      1.00       228
WALKING_DOWNSTAIRS       1.00      1.00      1.00       195
  WALKING_UPSTAIRS       1.00      1.00      1.00       215

          accuracy                           1.00       638
         macro avg       1.00      1.00      1.00       638
      weighted avg       1.00      1.00      1.00       638

Cohen's Kappa Score: 1.0000


In [ ]:
acc_df.loc[2, 'default_정확도(pca_교차검증)'] = np.mean(scores['test_accuracy'])
acc_df.loc[2, 'default_정확도(pca_테스트)'] = accuracy_score(y_test, y_pred)

In [ ]:
kappa_df.loc[2, 'default_kappa(pca_교차검증)'] = np.mean(scores['test_kappa'])
kappa_df.loc[2, 'default_kappa(pca_테스트)'] = cohen_kappa_score(y_test, y_pred)

In [ ]:
acc_df

,모델,default_정확도(교차검증),default_정확도(테스트),default_정확도(fi_교차검증),default_정확도(fi_테스트),default_정확도(pca_교차검증),default_정확도(pca_테스트)
0,RandomForest,0.984889,0.990596,0.987157,0.989028,0.961842,0.970219
1,LogisticRegression,0.995466,0.998433,0.995845,0.998433,0.993955,0.996865
2,SVM,0.993578,0.998433,0.995467,0.998433,0.998489,1.000000
3,DecisionTreeClassifier,0.929733,0.942006,0.933512,0.942006,NaN,NaN


In [ ]:
kappa_df

,모델,default_kappa(교차검증),default_kappa(테스트),default_kappa(fi_교차검증),default_kappa(fi_테스트),default_kappa(pca_교차검증),default_kappa(pca_테스트)
0,RandomForest,0.977231,0.985866,0.980644,0.983511,0.942435,0.955210
1,LogisticRegression,0.993165,0.997644,0.993736,0.997644,0.990889,0.995288
2,SVM,0.990319,0.997644,0.993166,0.997644,0.997723,1.000000
3,DecisionTreeClassifier,0.894093,0.912893,0.899815,0.912853,NaN,NaN


### 결정 트리

In [49]:
dt_model = DecisionTreeClassifier(random_state=42)
scores = cross_validate(dt_model, X_train_pca, y_train, cv=5, scoring={'accuracy': 'accuracy', 'kappa': kappa_scorer})
print(f"평균 정확도:", np.mean(scores['test_accuracy']))
print(f"평균 카파  :", np.mean(scores['test_kappa']))

평균 정확도: 0.8651325034775474
평균 카파  : 0.7967191981790636


In [ ]:
dt_model = DecisionTreeClassifier(random_state=42)
dt_model.fit(X_train_pca, y_train)

y_pred = dt_model.predict(X_test_pca)

print('accuracy :',accuracy_score(y_test, y_pred))
print('='*60)
print(confusion_matrix(y_test, y_pred))
print('='*60)
print(classification_report(y_test, y_pred))
print(f"Cohen's Kappa Score: {cohen_kappa_score(y_test, y_pred):.4f}")

accuracy : 0.8526645768025078
[[208   9  11]
 [ 12 166  17]
 [ 18  27 170]]
                    precision    recall  f1-score   support

           WALKING       0.87      0.91      0.89       228
WALKING_DOWNSTAIRS       0.82      0.85      0.84       195
  WALKING_UPSTAIRS       0.86      0.79      0.82       215

          accuracy                           0.85       638
         macro avg       0.85      0.85      0.85       638
      weighted avg       0.85      0.85      0.85       638

Cohen's Kappa Score: 0.7786


In [ ]:
acc_df.loc[3, 'default_정확도(pca_교차검증)'] = np.mean(scores['test_accuracy'])
acc_df.loc[3, 'default_정확도(pca_테스트)'] = accuracy_score(y_test, y_pred)

In [ ]:
kappa_df.loc[3, 'default_kappa(pca_교차검증)'] = np.mean(scores['test_kappa'])
kappa_df.loc[3, 'default_kappa(pca_테스트)'] = cohen_kappa_score(y_test, y_pred)

In [ ]:
acc_df

,모델,default_정확도(교차검증),default_정확도(테스트),default_정확도(fi_교차검증),default_정확도(fi_테스트),default_정확도(pca_교차검증),default_정확도(pca_테스트)
0,RandomForest,0.984889,0.990596,0.987157,0.989028,0.961842,0.970219
1,LogisticRegression,0.995466,0.998433,0.995845,0.998433,0.993955,0.996865
2,SVM,0.993578,0.998433,0.995467,0.998433,0.998489,1.000000
3,DecisionTreeClassifier,0.929733,0.942006,0.933512,0.942006,0.865133,0.852665


In [ ]:
kappa_df

,모델,default_kappa(교차검증),default_kappa(테스트),default_kappa(fi_교차검증),default_kappa(fi_테스트),default_kappa(pca_교차검증),default_kappa(pca_테스트)
0,RandomForest,0.977231,0.985866,0.980644,0.983511,0.942435,0.955210
1,LogisticRegression,0.993165,0.997644,0.993736,0.997644,0.990889,0.995288
2,SVM,0.990319,0.997644,0.993166,0.997644,0.997723,1.000000
3,DecisionTreeClassifier,0.894093,0.912893,0.899815,0.912853,0.796719,0.778554


In [ ]:
acc_df.to_csv('모델 정확도(디폴트_fi_pca).csv', index=False, encoding='utf-8-sig')

In [ ]:
kappa_df.to_csv('모델 kappa(디폴트_fi_pca).csv', index=False, encoding='utf-8-sig')

##방법3. permutation_importance()

In [ ]:
acc_df = pd.read_csv('모델 정확도(디폴트_fi_pca).csv')
acc_df

,모델,default_정확도(교차검증),default_정확도(테스트),default_정확도(fi_교차검증),default_정확도(fi_테스트),default_정확도(pca_교차검증),default_정확도(pca_테스트)
0,RandomForest,0.984889,0.990596,0.987157,0.989028,0.961842,0.970219
1,LogisticRegression,0.995466,0.998433,0.995845,0.998433,0.993955,0.996865
2,SVM,0.993578,0.998433,0.995467,0.998433,0.998489,1.000000
3,DecisionTreeClassifier,0.929733,0.942006,0.933512,0.942006,0.865133,0.852665


In [ ]:
rf_model = joblib.load('rf_model_default.pkl')
lr_model = joblib.load('lr_model_default.pkl')
svm_model = joblib.load('svm_model_default.pkl')
dt_model = joblib.load('dt_model_default.pkl')

### 랜덤포레스트

In [ ]:
model = rf_model

# permutation importance 계산
result_rf = permutation_importance(model, X_test, y_test, n_repeats=30, random_state=42, n_jobs=-1)

In [ ]:
# 중요도 정리 + 누적 중요도 계산
importances_df = pd.DataFrame({
    'feature': X_train.columns,
    'importance': result_rf.importances_mean
}).sort_values(by='importance', ascending=False)

# 중요도가 양수인 피처만 선택
importances_df = importances_df[importances_df['importance'] > 0]

print(f"선택된 피처 수: {len(importances_df)}")

# 피처 이름 추출
selected_features = importances_df['feature']

# 선택된 피처로 데이터 재구성
X_train_sel = X_train[selected_features]
X_test_sel = X_test[selected_features]

선택된 피처 수: 70


In [ ]:
importances_df.to_csv('rf_permutation_importance_DataFrame(디폴트).csv', index=False, encoding='utf-8-sig')

In [ ]:
rf_model = RandomForestClassifier(random_state=42)
scores = cross_validate(rf_model, X_train_sel, y_train, cv=5, scoring={'accuracy': 'accuracy', 'kappa': kappa_scorer})
print(f"평균 정확도:", np.mean(scores['test_accuracy']))
print(f"평균 카파  :", np.mean(scores['test_kappa']))

평균 정확도: 0.981866105503442
평균 카파  : 0.9726733787303173


In [ ]:
rf_model = RandomForestClassifier(random_state=42)

rf_model.fit(X_train_sel, y_train)
y_pred = rf_model.predict(X_test_sel)

print('accuracy :',accuracy_score(y_test, y_pred))
print('='*60)
print(confusion_matrix(y_test, y_pred))
print('='*60)
print(classification_report(y_test, y_pred))
print(f"Cohen's Kappa Score: {cohen_kappa_score(y_test, y_pred):.4f}")

accuracy : 0.9827586206896551
[[222   0   6]
 [  0 192   3]
 [  0   2 213]]
                    precision    recall  f1-score   support

           WALKING       1.00      0.97      0.99       228
WALKING_DOWNSTAIRS       0.99      0.98      0.99       195
  WALKING_UPSTAIRS       0.96      0.99      0.97       215

          accuracy                           0.98       638
         macro avg       0.98      0.98      0.98       638
      weighted avg       0.98      0.98      0.98       638

Cohen's Kappa Score: 0.9741


In [ ]:
acc_df.loc[0, 'default_정확도(pi_교차검증)'] = np.mean(scores['test_accuracy'])
acc_df.loc[0, 'default_정확도(pi_테스트)'] = accuracy_score(y_test, y_pred)

In [ ]:
kappa_df.loc[0, 'default_kappa(pi_교차검증)'] = np.mean(scores['test_kappa'])
kappa_df.loc[0, 'default_kappa(pi_테스트)'] = cohen_kappa_score(y_test, y_pred)

In [ ]:
acc_df

,모델,default_정확도(교차검증),default_정확도(테스트),default_정확도(fi_교차검증),default_정확도(fi_테스트),default_정확도(pca_교차검증),default_정확도(pca_테스트),default_정확도(pi_교차검증),default_정확도(pi_테스트)
0,RandomForest,0.984889,0.990596,0.987157,0.989028,0.961842,0.970219,0.981866,0.982759
1,LogisticRegression,0.995466,0.998433,0.995845,0.998433,0.993955,0.996865,NaN,NaN
2,SVM,0.993578,0.998433,0.995467,0.998433,0.998489,1.000000,NaN,NaN
3,DecisionTreeClassifier,0.929733,0.942006,0.933512,0.942006,0.865133,0.852665,NaN,NaN


In [ ]:
kappa_df

,모델,default_kappa(교차검증),default_kappa(테스트),default_kappa(fi_교차검증),default_kappa(fi_테스트),default_kappa(pca_교차검증),default_kappa(pca_테스트),default_kappa(pi_교차검증),default_kappa(pi_테스트)
0,RandomForest,0.977231,0.985866,0.980644,0.983511,0.942435,0.955210,0.972673,0.974091
1,LogisticRegression,0.993165,0.997644,0.993736,0.997644,0.990889,0.995288,NaN,NaN
2,SVM,0.990319,0.997644,0.993166,0.997644,0.997723,1.000000,NaN,NaN
3,DecisionTreeClassifier,0.894093,0.912893,0.899815,0.912853,0.796719,0.778554,NaN,NaN


### 로지스틱회귀

In [ ]:
model = lr_model

# permutation importance 계산
result = permutation_importance(model, X_test, y_test, n_repeats=30, random_state=42, n_jobs=-1)

In [ ]:
# 중요도 정리 + 누적 중요도 계산
importances_df = pd.DataFrame({
    'feature': X_train.columns,
    'importance': result.importances_mean
}).sort_values(by='importance', ascending=False)

# 중요도가 양수인 피처만 선택
importances_df = importances_df[importances_df['importance'] > 0]

print(f"선택된 피처 수: {len(importances_df)}")

# 피처 이름 추출
selected_features = importances_df['feature']

# 선택된 피처로 데이터 재구성
X_train_sel = X_train[selected_features]
X_test_sel = X_test[selected_features]

선택된 피처 수: 59


In [ ]:
importances_df.to_csv('lr_permutation_importance_DataFrame(디폴트).csv', index=False, encoding='utf-8-sig')

In [ ]:
lr_model = LogisticRegression(max_iter=1000, random_state=42)
scores = cross_validate(lr_model, X_train_sel, y_train, cv=5, scoring={'accuracy': 'accuracy', 'kappa': kappa_scorer})
print(f"평균 정확도:", np.mean(scores['test_accuracy']))
print(f"평균 카파  :", np.mean(scores['test_kappa']))

평균 정확도: 0.9928209152191746
평균 카파  : 0.989176809883569


In [ ]:
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train_sel, y_train)
y_pred = lr_model.predict(X_test_sel)

print('accuracy :',accuracy_score(y_test, y_pred))
print('='*60)
print(confusion_matrix(y_test, y_pred))
print('='*60)
print(classification_report(y_test, y_pred))
print(f"Cohen's Kappa Score: {cohen_kappa_score(y_test, y_pred):.4f}")

accuracy : 0.9968652037617555
[[227   0   1]
 [  1 194   0]
 [  0   0 215]]
                    precision    recall  f1-score   support

           WALKING       1.00      1.00      1.00       228
WALKING_DOWNSTAIRS       1.00      0.99      1.00       195
  WALKING_UPSTAIRS       1.00      1.00      1.00       215

          accuracy                           1.00       638
         macro avg       1.00      1.00      1.00       638
      weighted avg       1.00      1.00      1.00       638

Cohen's Kappa Score: 0.9953


In [ ]:
acc_df.loc[1, 'default_정확도(pi_교차검증)'] = np.mean(scores['test_accuracy'])
acc_df.loc[1, 'default_정확도(pi_테스트)'] = accuracy_score(y_test, y_pred)

In [ ]:
kappa_df.loc[1, 'default_kappa(pi_교차검증)'] = np.mean(scores['test_kappa'])
kappa_df.loc[1, 'default_kappa(pi_테스트)'] = cohen_kappa_score(y_test, y_pred)

In [ ]:
acc_df

,모델,default_정확도(교차검증),default_정확도(테스트),default_정확도(fi_교차검증),default_정확도(fi_테스트),default_정확도(pca_교차검증),default_정확도(pca_테스트),default_정확도(pi_교차검증),default_정확도(pi_테스트)
0,RandomForest,0.984889,0.990596,0.987157,0.989028,0.961842,0.970219,0.981866,0.982759
1,LogisticRegression,0.995466,0.998433,0.995845,0.998433,0.993955,0.996865,0.992821,0.996865
2,SVM,0.993578,0.998433,0.995467,0.998433,0.998489,1.000000,NaN,NaN
3,DecisionTreeClassifier,0.929733,0.942006,0.933512,0.942006,0.865133,0.852665,NaN,NaN


In [ ]:
kappa_df

,모델,default_kappa(교차검증),default_kappa(테스트),default_kappa(fi_교차검증),default_kappa(fi_테스트),default_kappa(pca_교차검증),default_kappa(pca_테스트),default_kappa(pi_교차검증),default_kappa(pi_테스트)
0,RandomForest,0.977231,0.985866,0.980644,0.983511,0.942435,0.955210,0.972673,0.974091
1,LogisticRegression,0.993165,0.997644,0.993736,0.997644,0.990889,0.995288,0.989177,0.995288
2,SVM,0.990319,0.997644,0.993166,0.997644,0.997723,1.000000,NaN,NaN
3,DecisionTreeClassifier,0.894093,0.912893,0.899815,0.912853,0.796719,0.778554,NaN,NaN


### SVM

In [ ]:
model = svm_model

# permutation importance 계산
result = permutation_importance(model, X_test, y_test, n_repeats=10, random_state=42, n_jobs=-1) # n_repeats=30 -> 너무 오래 걸림...30분이 넘게 걸려 결국 포기.  n_repeats=10 으로 실행
# 너무 오래 걸려서 안되면.... permutation_importance()를 랜덤포레스트로 하고 훈련을 svm으로 시키는 것을 고려할 것임.

In [ ]:
# 중요도 정리 + 누적 중요도 계산
importances_df = pd.DataFrame({
    'feature': X_train.columns,
    'importance': result.importances_mean
}).sort_values(by='importance', ascending=False)

# 중요도가 양수인 피처만 선택
importances_df = importances_df[importances_df['importance'] > 0]

print(f"선택된 피처 수: {len(importances_df)}")

# 피처 이름 추출
selected_features = importances_df['feature']

# 선택된 피처로 데이터 재구성
X_train_sel = X_train[selected_features]
X_test_sel = X_test[selected_features]

선택된 피처 수: 14


In [ ]:
importances_df.to_csv('svm_permutation_importance_DataFrame(디폴트).csv', index=False, encoding='utf-8-sig')

In [ ]:
svm_model = SVC(random_state=42)
scores = cross_validate(svm_model, X_train_sel, y_train, cv=5, scoring={'accuracy': 'accuracy', 'kappa': kappa_scorer})
print(f"평균 정확도:", np.mean(scores['test_accuracy']))
print(f"평균 카파  :", np.mean(scores['test_kappa']))

평균 정확도: 0.9357741555801262
평균 카파  : 0.9031384312308551


In [ ]:
svm_model = SVC(random_state=42)
svm_model.fit(X_train_sel, y_train)

y_pred = svm_model.predict(X_test_sel)

print('accuracy :',accuracy_score(y_test, y_pred))
print('='*60)
print(confusion_matrix(y_test, y_pred))
print('='*60)
print(classification_report(y_test, y_pred))
print(f"Cohen's Kappa Score: {cohen_kappa_score(y_test, y_pred):.4f}")

accuracy : 0.9467084639498433
[[223   2   3]
 [  6 176  13]
 [  1   9 205]]
                    precision    recall  f1-score   support

           WALKING       0.97      0.98      0.97       228
WALKING_DOWNSTAIRS       0.94      0.90      0.92       195
  WALKING_UPSTAIRS       0.93      0.95      0.94       215

          accuracy                           0.95       638
         macro avg       0.95      0.94      0.95       638
      weighted avg       0.95      0.95      0.95       638

Cohen's Kappa Score: 0.9198


In [ ]:
acc_df.loc[2, 'default_정확도(pi_교차검증)'] = np.mean(scores['test_accuracy'])
acc_df.loc[2, 'default_정확도(pi_테스트)'] = accuracy_score(y_test, y_pred)

In [ ]:
kappa_df.loc[2, 'default_kappa(pi_교차검증)'] = np.mean(scores['test_kappa'])
kappa_df.loc[2, 'default_kappa(pi_테스트)'] = cohen_kappa_score(y_test, y_pred)

In [ ]:
acc_df

,모델,default_정확도(교차검증),default_정확도(테스트),default_정확도(fi_교차검증),default_정확도(fi_테스트),default_정확도(pca_교차검증),default_정확도(pca_테스트),default_정확도(pi_교차검증),default_정확도(pi_테스트)
0,RandomForest,0.984889,0.990596,0.987157,0.989028,0.961842,0.970219,0.981866,0.982759
1,LogisticRegression,0.995466,0.998433,0.995845,0.998433,0.993955,0.996865,0.992821,0.996865
2,SVM,0.993578,0.998433,0.995467,0.998433,0.998489,1.000000,0.935774,0.946708
3,DecisionTreeClassifier,0.929733,0.942006,0.933512,0.942006,0.865133,0.852665,NaN,NaN


In [ ]:
kappa_df

,모델,default_kappa(교차검증),default_kappa(테스트),default_kappa(fi_교차검증),default_kappa(fi_테스트),default_kappa(pca_교차검증),default_kappa(pca_테스트),default_kappa(pi_교차검증),default_kappa(pi_테스트)
0,RandomForest,0.977231,0.985866,0.980644,0.983511,0.942435,0.955210,0.972673,0.974091
1,LogisticRegression,0.993165,0.997644,0.993736,0.997644,0.990889,0.995288,0.989177,0.995288
2,SVM,0.990319,0.997644,0.993166,0.997644,0.997723,1.000000,0.903138,0.919845
3,DecisionTreeClassifier,0.894093,0.912893,0.899815,0.912853,0.796719,0.778554,NaN,NaN


### 결정트리

In [ ]:
model = dt_model

# permutation importance 계산
result = permutation_importance(model, X_test, y_test, n_repeats=30, random_state=42, n_jobs=-1)

In [ ]:
# 중요도 정리 + 누적 중요도 계산
importances_df = pd.DataFrame({
    'feature': X_train.columns,
    'importance': result.importances_mean
}).sort_values(by='importance', ascending=False)

# 중요도가 양수인 피처만 선택
importances_df = importances_df[importances_df['importance'] > 0]

print(f"선택된 피처 수: {len(importances_df)}")

# 피처 이름 추출
selected_features = importances_df['feature']

# 선택된 피처로 데이터 재구성
X_train_sel = X_train[selected_features]
X_test_sel = X_test[selected_features]

선택된 피처 수: 51


In [ ]:
importances_df.to_csv('dt_permutation_importance_DataFrame(디폴트).csv', index=False, encoding='utf-8-sig')

In [ ]:
dt_model = DecisionTreeClassifier(random_state=42)
scores = cross_validate(dt_model, X_train_sel, y_train, cv=5, scoring={'accuracy': 'accuracy', 'kappa': kappa_scorer})
print(f"평균 정확도:", np.mean(scores['test_accuracy']))
print(f"평균 카파  :", np.mean(scores['test_kappa']))

평균 정확도: 0.947485822306238
평균 카파  : 0.9208616084394748


In [ ]:
dt_model = DecisionTreeClassifier( random_state=42)
dt_model.fit(X_train_sel, y_train)

y_pred = dt_model.predict(X_test_sel)

print('accuracy :',accuracy_score(y_test, y_pred))
print('='*60)
print(confusion_matrix(y_test, y_pred))
print('='*60)
print(classification_report(y_test, y_pred))
print(f"Cohen's Kappa Score: {cohen_kappa_score(y_test, y_pred):.4f}")

accuracy : 0.9498432601880877
[[214   1  13]
 [  0 189   6]
 [  6   6 203]]
                    precision    recall  f1-score   support

           WALKING       0.97      0.94      0.96       228
WALKING_DOWNSTAIRS       0.96      0.97      0.97       195
  WALKING_UPSTAIRS       0.91      0.94      0.93       215

          accuracy                           0.95       638
         macro avg       0.95      0.95      0.95       638
      weighted avg       0.95      0.95      0.95       638

Cohen's Kappa Score: 0.9246


In [ ]:
acc_df.loc[3, 'default_정확도(pi_교차검증)'] = np.mean(scores['test_accuracy'])
acc_df.loc[3, 'default_정확도(pi_테스트)'] = accuracy_score(y_test, y_pred)

In [ ]:
kappa_df.loc[3, 'default_kappa(pi_교차검증)'] = np.mean(scores['test_kappa'])
kappa_df.loc[3, 'default_kappa(pi_테스트)'] = cohen_kappa_score(y_test, y_pred)

In [ ]:
acc_df

,모델,default_정확도(교차검증),default_정확도(테스트),default_정확도(fi_교차검증),default_정확도(fi_테스트),default_정확도(pca_교차검증),default_정확도(pca_테스트),default_정확도(pi_교차검증),default_정확도(pi_테스트)
0,RandomForest,0.984889,0.990596,0.987157,0.989028,0.961842,0.970219,0.981866,0.982759
1,LogisticRegression,0.995466,0.998433,0.995845,0.998433,0.993955,0.996865,0.992821,0.996865
2,SVM,0.993578,0.998433,0.995467,0.998433,0.998489,1.000000,0.935774,0.946708
3,DecisionTreeClassifier,0.929733,0.942006,0.933512,0.942006,0.865133,0.852665,0.947486,0.949843


In [ ]:
kappa_df

,모델,default_kappa(교차검증),default_kappa(테스트),default_kappa(fi_교차검증),default_kappa(fi_테스트),default_kappa(pca_교차검증),default_kappa(pca_테스트),default_kappa(pi_교차검증),default_kappa(pi_테스트)
0,RandomForest,0.977231,0.985866,0.980644,0.983511,0.942435,0.955210,0.972673,0.974091
1,LogisticRegression,0.993165,0.997644,0.993736,0.997644,0.990889,0.995288,0.989177,0.995288
2,SVM,0.990319,0.997644,0.993166,0.997644,0.997723,1.000000,0.903138,0.919845
3,DecisionTreeClassifier,0.894093,0.912893,0.899815,0.912853,0.796719,0.778554,0.920862,0.924646


In [ ]:
acc_df.to_csv('모델 정확도(디폴트_fi_pca_pi).csv', index=False, encoding='utf-8-sig')

In [ ]:
kappa_df.to_csv('모델 kappa(디폴트_fi_pca_pi).csv', index=False, encoding='utf-8-sig')